# SAA-Lite — Buoc 1: profiling 4 cau hinh

Muc dich: chon ra cau hinh Lite thang cuoc truoc khi tieu gio GPU cho Buoc 2.

Chay 4 cau hinh tren MOT class (`carpet`, full 117 anh), do latency tach theo
giai doan DINO / SAM / saliency.

| Cau hinh | SAM | Saliency |
|---|---|---|
| baseline | SAM ViT-H (2.4 GB) | WideResNet-50 |
| lite1 | MobileSAM | WideResNet-50 |
| lite2 | MobileSAM | MobileNetV3 |
| lite3 | EfficientViT-SAM-L0 | WideResNet-50 |

Tach tung bien mot de biet chinh xac module nao gay mat accuracy.

Notebook nay doc lap voi `Benchmark_SAA.ipynb` (chay baseline day du).

In [ ]:
# Cai dat. An toan khi chay lai nhieu lan.
%cd /content

# Xoa clone cu TRUOC khi clone. Khong co dong nay thi git clone bao
# "destination path already exists", bo qua im lang, va ban chay tiep bang
# code cu ma khong biet.
!rm -rf /content/Segment-Any-Anomaly
!git clone -b dev https://github.com/SyDuc7421/Segment-Any-Anomaly.git
%cd Segment-Any-Anomaly/

# setuptools >= 80 da bo lenh `setup.py develop`, ma pip dung dung lenh do cho
# ban editable khi co --no-build-isolation. Colab nang image len la GroundingDINO
# gay voi "python setup.py develop did not run successfully". Ghim lai truoc.
!pip install -q "setuptools<80" wheel
import setuptools
print('setuptools:', setuptools.__version__)

# Go pin transformers<4.36 cua GroundingDINO. Phai quet CA requirements.txt,
# khong chi *.py: pip doc requirements.txt, va bo sot no thi pip ha transformers
# xuong 4.35 (keo theo huggingface_hub va tokenizers), roi lenh pip cuoi cell
# lai day len 5.x - vong xoay do de lai mot dong conflict gia.
# Code GroundingDINO trong repo nay da duoc va cho transformers 5.x tu truoc.
import re, pathlib

for pattern in ('*.py', '*.txt'):
    for p in pathlib.Path('GroundingDINO').rglob(pattern):
        txt = p.read_text()
        patched = re.sub(r'transformers[^"\'\n]*<4\.\d+(\.\d+)?', 'transformers>=4.41.0', txt)
        if patched != txt:
            p.write_text(patched)
            print('go pin transformers trong', p)

# KHONG dat -q cho hai lenh editable duoi day: day la cho de gay nhat, va
# loi that nam trong phan output ma -q nuot mat.
%cd GroundingDINO/
!pip install -e . --no-build-isolation
%cd ../SAM
!pip install -e .
%cd ..

!pip install -q "transformers>=4.41.0" "supervision>=0.6.0,<0.21.0" \
    opencv-python pycocotools matplotlib onnxruntime onnx ipykernel gradio loguru

# KHONG kiem tra import o day. Ban editable ghi duong dan vao mot file .pth,
# ma .pth chi duoc doc luc interpreter khoi dong - package vua cai xong van
# "khong ton tai" voi kernel dang chay. Kiem tra nam o cell sau lenh restart.
print('\nCai dat xong. Chay cell tiep theo de restart runtime,')
print('roi cell sau do se xac nhan package da cai duoc that.')

In [ ]:
# Restart so updated transformers is loaded from disk
import os
os.kill(os.getpid(), 9)

In [ ]:
# Xac nhan package da cai THAT - chay sau restart, vi ban editable chi hien
# ra voi interpreter khoi dong lai.
%cd /content/Segment-Any-Anomaly
import importlib.util

missing = [m for m in ('groundingdino', 'segment_anything')
           if importlib.util.find_spec(m) is None]

for m in ('groundingdino', 'segment_anything'):
    print(f'{m}: {"THIEU" if m in missing else "OK"}')

if missing:
    raise RuntimeError(
        f'Cai dat that bai: {missing}. DUNG chay tiep - moi class se chet o dong '
        f'import. Doc output pip cua cell 1 de biet la loi setuptools hay loi '
        f'bien dich CUDA extension.'
    )

%cd /content/Segment-Any-Anomaly
%mkdir -p weights
%cd weights
# -nc: co file roi thi bo qua. Khong co no thi chay lai cell se tai lai 2.4 GB
# va de ra sam_vit_h_4b8939.pth.1 - mot ban sao vo dung.
!wget -nc -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
!wget -nc -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
%cd ..

In [ ]:
%cd /content/Segment-Any-Anomaly
%mkdir -p /content/datasets

from google.colab import userdata
import json, pathlib, os

pathlib.Path('/root/.kaggle').mkdir(exist_ok=True)
pathlib.Path('/root/.kaggle/kaggle.json').write_text(json.dumps({
    'username': userdata.get('KAGGLE_USERNAME'),
    'key': userdata.get('KAGGLE_KEY')
}))
!chmod 600 /root/.kaggle/kaggle.json

# Accept dataset terms first at: https://www.kaggle.com/datasets/ipythonx/mvtec-ad
!pip install -q kaggle
!kaggle datasets download -d ipythonx/mvtec-ad -p /content/datasets/ --unzip

os.environ['MVTEC_DIR'] = '/content/datasets'

from datasets import mvtec_classes
present = [c for c in mvtec_classes if os.path.isdir(f'/content/datasets/{c}')]
print(f'MVTec: {len(present)}/15 classes ready:', present)

## Cai backbone nhe

In [ ]:
# MobileSAM cho Lite-1 va Lite-2. No giu nguyen interface SamPredictor nen la
# drop-in that su - SAA/backbones.py khong phai doi gi ngoai ten bien the.
%cd /content/Segment-Any-Anomaly
!pip install -q git+https://github.com/ChaoningZhang/MobileSAM.git
!wget -q -P weights/ https://github.com/ChaoningZhang/MobileSAM/raw/master/weights/mobile_sam.pt

import os, importlib.util

ckpt = 'weights/mobile_sam.pt'
size_mb = os.path.getsize(ckpt) / 1e6 if os.path.exists(ckpt) else 0
print(f'{ckpt}: {"OK, %.0f MB" % size_mb if size_mb > 1 else "THIEU - kiem tra URL"}')
print('mobile_sam:', 'OK' if importlib.util.find_spec('mobile_sam') else 'THIEU')

# Cai them co the keo torch/transformers khac ve. Cho no gay O DAY chu dung de
# gay giua lan profiling.
try:
    from GroundingDINO.groundingdino.models import build_model
    print('GroundingDINO van OK sau khi cai')
except Exception as e:
    print(f'CANH BAO: GroundingDINO gay sau khi cai MobileSAM - {type(e).__name__}: {e}')

In [ ]:
# TUY CHON - chi can cho Lite-3. Bo qua cell nay cung duoc: hai cau hinh Lite
# con lai van du ve bieu do Pareto.
#
# Day la cho rui ro nhat: nhanh efficientvit_l0 trong SAA/backbones.py viet tu
# README cua mit-han-lab/efficientvit va CHUA TUNG chay. Pip install repo nay
# cung co the keo torch/transformers khac ve, lam gay GroundingDINO dang chay
# duoc. Hong qua mot tieng thi bo Lite-3.
%cd /content/Segment-Any-Anomaly
!pip install -q git+https://github.com/mit-han-lab/efficientvit.git
!wget -q -P weights/ https://huggingface.co/mit-han-lab/efficientvit-sam/resolve/main/efficientvit_sam_l0.pt

import os
ckpt = 'weights/efficientvit_sam_l0.pt'
size_mb = os.path.getsize(ckpt) / 1e6 if os.path.exists(ckpt) else 0
print(f'{ckpt}: {"OK, %.0f MB" % size_mb if size_mb > 1 else "THIEU - URL co the da doi"}')

try:
    from efficientvit.sam_model_zoo import create_sam_model
    from efficientvit.models.efficientvit.sam import EfficientViTSamPredictor
    print('import efficientvit: OK - hai ten trong SAA/backbones.py khop')
except Exception as e:
    print(f'import efficientvit: FAIL - {type(e).__name__}: {e}')
    print('Sua nhanh efficientvit_l0 trong SAA/backbones.py cho khop, hoac bo Lite-3.')

try:
    from GroundingDINO.groundingdino.models import build_model
    print('GroundingDINO van OK sau khi cai')
except Exception as e:
    print(f'CANH BAO: cai EfficientViT lam gay GroundingDINO - {type(e).__name__}: {e}')

## Gan Drive va chon class

In [ ]:
# Ghi thang vao Drive, tung cau hinh mot. Disconnect giua chung thi cau hinh
# da xong van con.
from google.colab import drive
import os

try:
    drive.mount('/content/drive')
except ValueError:
    # "Mountpoint must not already contain files": Drive da gan tu truoc trong
    # cung session, hoac con thu muc cu sot lai.
    drive.mount('/content/drive', force_remount=True)

PROF_ROOT = '/content/drive/MyDrive/SAA_results/prof'
CLASS_NAME = 'carpet'

os.environ['MVTEC_DIR'] = '/content/datasets'
print('PROF_ROOT  =', PROF_ROOT)
print('CLASS_NAME =', CLASS_NAME)

## Chay profiling

In [ ]:
# Goi thang eval_SAA.py chu khong qua run_MVTec.py: moi cau hinh can thu muc
# rieng, vi csv_path khong mang danh tinh cau hinh nen dung chung la ghi de.
%cd /content/Segment-Any-Anomaly
import os, subprocess, pandas as pd

CONFIGS = [
    ('baseline', 'vit_h',           'wide_resnet50', 'weights/sam_vit_h_4b8939.pth'),
    ('lite1',    'mobile_sam',      'wide_resnet50', 'weights/mobile_sam.pt'),
    ('lite2',    'mobile_sam',      'mobilenetv3',   'weights/mobile_sam.pt'),
    ('lite3',    'efficientvit_l0', 'wide_resnet50', 'weights/efficientvit_sam_l0.pt'),
]

def already_done(root):
    path = f'{root}/csv/mvtec-indx-0.csv'
    if not os.path.exists(path):
        return False
    df = pd.read_csv(path, index_col=0)
    return CLASS_NAME in df.index and df.loc[CLASS_NAME, 'p_ap'] > 0

# KHONG dat --max-samples: ca 4 cau hinh phai chay dung cung mot tap anh thi
# p_ap va p_f1 moi so duoc voi nhau (rang buoc normalize(), spec muc 2.7).
# --cal-pro False: profiling chi can latency, PRO va r_f1 ton CPU.
# --vis False: khong ghi hang tram anh qua Drive FUSE.
for name, sam, saliency, ckpt in CONFIGS:
    root = f'{PROF_ROOT}_{name}'

    if already_done(root):
        print(f'skip {name}: da co ket qua trong {root}')
        continue
    if not os.path.exists(ckpt):
        print(f'skip {name}: thieu checkpoint {ckpt}')
        continue

    print(f'\n=== {name}: sam={sam} saliency={saliency} ===')
    r = subprocess.run([
        'python', 'eval_SAA.py',
        '--dataset', 'mvtec', '--class-name', CLASS_NAME,
        '--cal-pro', 'False',
        '--vis', 'False',
        '--sam-variant', sam,
        '--saliency-backbone', saliency,
        '--sam_checkpoint', ckpt,
        '--root-dir', root,
    ], cwd='/content/Segment-Any-Anomaly')

    if r.returncode != 0:
        print(f'LOI: {name} thoat voi ma {r.returncode} - dung lai de doc loi o tren')
        break

## Ket qua

In [ ]:
# Ba quyet dinh rut ra tu bang nay:
#   1. Cau hinh Lite nao thang cuoc (tieu chi spec muc 7)
#   2. DINO co thanh nut co chai sau khi thay SAM khong (gia thuyet spec muc 4.4)
#   3. Cong cache DINO: full MVTec duoi ~1h thi bo cache han (spec muc 4.5)
import pandas as pd, glob, os

MVTEC_IMAGES = 1725   # tong so anh test cua 15 class MVTec

rows = {}
for p in sorted(glob.glob(f'{PROF_ROOT}_*/csv/mvtec-indx-0.csv')):
    name = os.path.basename(os.path.dirname(os.path.dirname(p))).replace('prof_', '')
    df = pd.read_csv(p, index_col=0)
    if CLASS_NAME in df.index and df.loc[CLASS_NAME, 'p_ap'] > 0:
        rows[name] = df.loc[CLASS_NAME]

if not rows:
    print('Chua co ket qua profiling nao.')
else:
    prof = pd.DataFrame(rows).T
    cols = [c for c in ['p_ap', 'p_f1', 't_dino', 't_sam', 't_saliency',
                        't_total', 'peak_vram', 'n_images'] if c in prof.columns]
    print(prof[cols].to_string(float_format='{:.2f}'.format))

    if 'baseline' not in prof.index:
        print('\nChua co baseline nen khong so sanh duoc.')
    else:
        base = prof.loc['baseline']

        print('\n--- so voi baseline (spec muc 7: >=3x nhanh, >=95% p_ap, >=95% p_f1) ---')
        for name in prof.index:
            if name == 'baseline':
                continue
            r = prof.loc[name]
            speedup = base['t_total'] / r['t_total'] if r['t_total'] else float('nan')
            ap = r['p_ap'] / base['p_ap'] * 100
            f1 = r['p_f1'] / base['p_f1'] * 100
            verdict = 'DAT' if (speedup >= 3 and ap >= 95 and f1 >= 95) else 'chua dat'
            print(f'{name:10s} {speedup:5.2f}x   p_ap {ap:5.1f}%   p_f1 {f1:5.1f}%   {verdict}')

        print('\n--- ty trong tung giai doan ---')
        for name in prof.index:
            r = prof.loc[name]
            if r['t_total']:
                print(f'{name:10s} dino {r["t_dino"]/r["t_total"]:5.0%}  '
                      f'sam {r["t_sam"]/r["t_total"]:5.0%}  '
                      f'saliency {r["t_saliency"]/r["t_total"]:5.0%}')

        print('\n--- cong cache DINO (spec muc 4.5) ---')
        for name in prof.index:
            hours = prof.loc[name, 't_total'] * MVTEC_IMAGES / 1000 / 3600
            gate = 'bo cache han' if hours < 1 else ('them cache' if hours > 2 else 'tuy')
            print(f'{name:10s} uoc full MVTec: {hours:4.1f}h -> {gate}')